In [20]:
from pathlib import Path
import sys

p = Path.cwd()
for _ in range(6):                       # climb up to find project root
    if (p / "config.py").exists():
        sys.path.insert(0, str(p))
        break
    p = p.parent

In [22]:
from config import *

In [ ]:
import os
import cv2
import numpy as np
from PIL import Image
from datetime import datetime
import argparse

try:
    import imagehash
except Exception:
    imagehash = None

try:
    from skimage.metrics import structural_similarity as ssim
except Exception:
    ssim = None

def _read_image_cv(path, flags=cv2.IMREAD_COLOR):
    img = cv2.imread(path, flags)
    if img is None:
        raise FileNotFoundError(f"Cannot read image: {path}")
    return img

def _to_gray(img):
    return cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

def _resize_to_min(img1, img2):
    h1, w1 = img1.shape[:2]
    h2, w2 = img2.shape[:2]
    h, w = min(h1, h2), min(w1, w2)
    return cv2.resize(img1, (w, h)), cv2.resize(img2, (w, h))

def mse(imgA, imgB):
    imgA, imgB = _resize_to_min(imgA, imgB)
    if imgA.ndim == 3:
        imgA = _to_gray(imgA)
        imgB = _to_gray(imgB)
    err = np.mean((imgA.astype("float") - imgB.astype("float")) ** 2)
    return float(err)

def structural_similarity(imgA, imgB):
    if ssim is None:
        raise RuntimeError("skimage is required for SSIM. Install scikit-image.")
    imgA, imgB = _resize_to_min(imgA, imgB)
    imgA = _to_gray(imgA)
    imgB = _to_gray(imgB)
    score, _ = ssim(imgA, imgB, full=True)
    return float(score)

def phash_similarity(pathA, pathB):
    if imagehash is None:
        raise RuntimeError("imagehash (and PIL) required for perceptual hash. Install imagehash.")
    ha = imagehash.phash(Image.open(pathA))
    hb = imagehash.phash(Image.open(pathB))
    # normalized similarity: 1 - (hamming / hash_size)
    max_bits = ha.hash.size
    hamming = (ha - hb)
    sim = 1.0 - (hamming / max_bits)
    return float(sim)

def hist_correlation(imgA, imgB, bins=32):
    imgA = cv2.cvtColor(imgA, cv2.COLOR_BGR2HSV)
    imgB = cv2.cvtColor(imgB, cv2.COLOR_BGR2HSV)
    imgA, imgB = _resize_to_min(imgA, imgB)
    histA = cv2.calcHist([imgA], [0, 1], None, [bins, bins], [0, 180, 0, 256])
    histB = cv2.calcHist([imgB], [0, 1], None, [bins, bins], [0, 180, 0, 256])
    cv2.normalize(histA, histA)
    cv2.normalize(histB, histB)
    score = cv2.compareHist(histA, histB, cv2.HISTCMP_CORREL)
    return float(score)

def orb_match_ratio(imgA, imgB, max_features=500):
    imgA_gray = _to_gray(imgA)
    imgB_gray = _to_gray(imgB)
    orb = cv2.ORB_create(nfeatures=max_features)
    kp1, des1 = orb.detectAndCompute(imgA_gray, None)
    kp2, des2 = orb.detectAndCompute(imgB_gray, None)
    if des1 is None or des2 is None:
        return 0.0
    bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
    matches = bf.match(des1, des2)
    if not matches:
        return 0.0
    # ratio of good matches to min(keypoints)
    ratio = len(matches) / max(1, min(len(kp1), len(kp2)))
    return float(ratio)

def compare_images(pathA, pathB):
    if not os.path.exists(pathA) or not os.path.exists(pathB):
        raise FileNotFoundError("One or both image paths do not exist.")
    imgA = _read_image_cv(pathA, cv2.IMREAD_COLOR)
    imgB = _read_image_cv(pathB, cv2.IMREAD_COLOR)

    results = {}
    results["mse"] = mse(imgA, imgB)                      # lower is more similar

    if ssim is not None:
        results["ssim"] = structural_similarity(imgA, imgB)   # 1.0 is identical
    else:
        results["ssim"] = None
    try:
        results["phash"] = phash_similarity(pathA, pathB)     # 1.0 is identical
    except Exception:
        results["phash"] = None
    results["hist_correlation"] = hist_correlation(imgA, imgB)  # 1.0 is identical
    results["orb_match_ratio"] = orb_match_ratio(imgA, imgB)    # 0..1 ratio

    return results



In [3]:
test_directory = "../data/test_data"

In [26]:
import os

img1 = rf"{TEST_DATA}/{LANGUAGES[0]}/Centuries/07.jpg"
img2 = rf"{TEST_DATA}/{LANGUAGES[1]}/Centuries/07.jpg"

def main():
    # Optional: set a log file path to append results, or None to skip logging
    log_path = r"d:\path\to\compare_history.txt"  # or None

    res = compare_images(img1, img2)
    now = datetime.now().isoformat()
    out_lines = [f"{now} - Compare {img1} VS {img2}"]
    for k, v in res.items():
        out_lines.append(f"  {k}: {v}")
    output = "\n".join(out_lines)
    print(output)

    if log_path:
        os.makedirs(os.path.dirname(log_path), exist_ok=True)
        with open(log_path, "a", encoding="utf-8") as f:
            f.write(output + "\n")

if __name__ == "__main__":
    main()

2026-03-16T04:10:02.929177 - Compare D:\Downloads\Tu_Lieu\Cao_Hoc\Master_Thesis\master-thesis\data\test_data/en/Centuries/07.jpg VS D:\Downloads\Tu_Lieu\Cao_Hoc\Master_Thesis\master-thesis\data\test_data/vi/Centuries/07.jpg
  mse: 781.4463481385031
  ssim: 0.9420246663911295
  phash: None
  hist_correlation: 1.0
  orb_match_ratio: 0.594


Result explain:

# 1: Similar image
```
  mse: 781.4463481385031
  ssim: 0.9420246663911295
  phash: None
  hist_correlation: 1.0
  orb_match_ratio: 0.594
```
Visually the two images are very similar (high SSIM, perfect histogram, many ORB matches).
They are not pixel-identical (non-zero MSE and PSNR ~19 dB), so there are noticeable pixel-level or compression/registration differences (cropping, small shifts, re-encoding, different compression artifacts, resized/resampled pixels, or watermarks).

# 2: Not similar image
```
  mse: 12173.020035068292
  ssim: 0.22791087477198066
  phash: None
  hist_correlation: 0.8188110791435751
  orb_match_ratio: 0.212
```
Images are largely dissimilar (high MSE, low SSIM, low ORB), though color distribution is somewhat similar (histogram). Likely causes: different content, heavy cropping/rotation/scale/translation, or strong visual edits (overlays, filters, compression).